# 00 - Contexto e inventario dos dados

## tl;dr
Este notebook inventaria os arquivos brutos do projeto Quatro Norte, confirma o modelo estrela e registra os principais riscos iniciais antes da preparacao analitica.

## Context & Methods

O objetivo do projeto e analisar historico de manutencao de carretas e prever custo interno de manutencao por km. A fonte local e a extracao `data/extract_custo_interno_km.sql`, materializada em sete CSVs em `data/raw`.

### Key Assumptions
- Os arquivos em `data/raw` sao a camada bruta e nao devem ser alterados manualmente.
- O identificador comum entre as bases e `ID_CARRETA`.
- A variavel-alvo sera derivada em etapa posterior, pois ainda nao existe pronta nos CSVs.

In [21]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))
(PROJECT_ROOT / ".cache" / "matplotlib").mkdir(parents=True, exist_ok=True)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
FIGURES = REPORTS / "figures"
TABLES = REPORTS / "tables"
ANALYSIS_START = pd.Timestamp("2020-01-01")
ANALYSIS_END_EXCLUSIVE = pd.Timestamp("2026-01-01")
ANALYSIS_END = pd.Timestamp("2025-12-31")
KM_MIN_MES_ALVO = 500.0
PREVENTIVE_VMRS_CODES = {"PM"}

for path in [DATA_INTERIM, DATA_PROCESSED, FIGURES, TABLES]:
    path.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

FILES = {
    "dim_carretas": DATA_RAW / "dim_carretas_2020-01-01_to_2025-12-31.csv",
    "fato_contratos": DATA_RAW / "fato_contratos_2020-01-01_to_2025-12-31.csv",
    "fato_gps": DATA_RAW / "fato_gps_2020-01-01_to_2025-12-31.csv",
    "fato_readings": DATA_RAW / "fato_readings_2020-01-01_to_2025-12-31.csv",
    "fato_wo": DATA_RAW / "fato_wo_2020-01-01_to_2025-12-31.csv",
    "fato_wo_labour": DATA_RAW / "fato_wo_labour_2020-01-01_to_2025-12-31.csv",
    "fato_wo_parts": DATA_RAW / "fato_wo_parts_2020-01-01_to_2025-12-31.csv",
}

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def read_csv(name: str, **kwargs) -> pd.DataFrame:
    return normalize_columns(pd.read_csv(FILES[name], low_memory=False, **kwargs))

def month_start(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce").dt.to_period("M").dt.to_timestamp()

def as_number(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

def mode_or_unknown(series: pd.Series, unknown: str = "SEM_INFORMACAO") -> str:
    values = series.dropna().astype(str)
    if values.empty:
        return unknown
    return values.mode().iloc[0]

## Data

### 1. Listar arquivos disponiveis

In [22]:
file_inventory = []
for name, path in FILES.items():
    header = pd.read_csv(path, nrows=0)
    file_inventory.append({
        "base": name,
        "arquivo": path.name,
        "tamanho_mb": path.stat().st_size / 1024**2,
        "n_colunas": len(header.columns),
        "colunas": ", ".join(header.columns),
    })

file_inventory = pd.DataFrame(file_inventory)
file_inventory

,base,arquivo,tamanho_mb,n_colunas,colunas
0,dim_carretas,dim_carretas_2020-01-01_to_2025-12-31.csv,1.661282,18,"ID_CARRETA, DESCRICAO_CARRETA, CHASSI_VIN, PLA..."
1,fato_contratos,fato_contratos_2020-01-01_to_2025-12-31.csv,2.889145,10,"ID_CONTRATO_CARRETA, ID_CARRETA, TIPO_CONTRATO..."
2,fato_gps,fato_gps_2020-01-01_to_2025-12-31.csv,25.249737,7,"ID_CARRETA, ID_BEACON, DATA_HORA_GPS, LATITUDE..."
3,fato_readings,fato_readings_2020-01-01_to_2025-12-31.csv,18.190387,8,"ID_LEITURA, ID_CARRETA, DATA_LEITURA, KM_ACUMU..."
4,fato_wo,fato_wo_2020-01-01_to_2025-12-31.csv,42.293529,12,"ID_OS, ID_CARRETA, NUMERO_OS, DATA_OS, SOLICIT..."
5,fato_wo_labour,fato_wo_labour_2020-01-01_to_2025-12-31.csv,77.957802,12,"ID_LINHA_MAO_OBRA, ID_OS, ID_CARRETA, DATA_OS,..."
6,fato_wo_parts,fato_wo_parts_2020-01-01_to_2025-12-31.csv,99.054669,13,"ID_LINHA_PECA, ID_OS, ID_CARRETA, DATA_OS, ID_..."


### 2. Medir volumes, chaves e cobertura temporal

In [23]:
date_columns = {
    "dim_carretas": ["DATA_ENTRADA_SERVICO"],
    "fato_contratos": ["DATA_INICIO", "DATA_FIM"],
    "fato_gps": ["DATA_HORA_GPS"],
    "fato_readings": ["DATA_LEITURA"],
    "fato_wo": ["DATA_OS"],
    "fato_wo_labour": ["DATA_OS"],
    "fato_wo_parts": ["DATA_OS"],
}

primary_keys = {
    "dim_carretas": "ID_CARRETA",
    "fato_contratos": "ID_CONTRATO_CARRETA",
    "fato_gps": None,
    "fato_readings": "ID_LEITURA",
    "fato_wo": "ID_OS",
    "fato_wo_labour": "ID_LINHA_MAO_OBRA",
    "fato_wo_parts": "ID_LINHA_PECA",
}

rows = []
for name, path in FILES.items():
    df = pd.read_csv(path, low_memory=False)
    row = {
        "base": name,
        "linhas": len(df),
        "colunas": len(df.columns),
        "grao": {
            "dim_carretas": "uma carreta",
            "fato_readings": "uma leitura de odometro",
            "fato_wo": "uma ordem de servico",
            "fato_wo_labour": "uma linha de mao de obra",
            "fato_wo_parts": "uma linha de peca",
            "fato_contratos": "uma carreta-contrato",
            "fato_gps": "um ponto GPS por carreta/dia",
        }[name],
    }
    if "ID_CARRETA" in df.columns:
        row["carretas_distintas"] = df["ID_CARRETA"].nunique(dropna=True)
    pk = primary_keys[name]
    if pk:
        row["chave_primaria"] = pk
        row["pk_distintas"] = df[pk].nunique(dropna=True)
        row["pk_duplicadas"] = len(df) - df[pk].nunique(dropna=True)
    for col in date_columns[name]:
        if col in df.columns:
            parsed = pd.to_datetime(df[col], errors="coerce")
            row[f"{col.lower()}_min"] = parsed.min()
            row[f"{col.lower()}_max"] = parsed.max()
            row[f"{col.lower()}_nulos"] = parsed.isna().sum()
    rows.append(row)

inventory = pd.DataFrame(rows)
inventory.to_csv(TABLES / "00_inventario_bases.csv", index=False)
inventory

,base,linhas,colunas,grao,carretas_distintas,chave_primaria,pk_distintas,pk_duplicadas,data_entrada_servico_min,data_entrada_servico_max,data_entrada_servico_nulos,data_inicio_min,data_inicio_max,data_inicio_nulos,data_fim_min,data_fim_max,data_fim_nulos,data_hora_gps_min,data_hora_gps_max,data_hora_gps_nulos,data_leitura_min,data_leitura_max,data_leitura_nulos,data_os_min,data_os_max,data_os_nulos
0,dim_carretas,10412,18,uma carreta,10412,ID_CARRETA,10412.0,0.0,1993-01-01,2024-09-24,5006.0,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN
1,fato_contratos,32232,10,uma carreta-contrato,10349,ID_CONTRATO_CARRETA,32232.0,0.0,NaT,NaT,NaN,2005-05-12,2026-04-14 13:43:00,2.0,2015-01-01 09:19:33,2026-03-31 22:14:00,9895.0,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN
2,fato_gps,227795,7,um ponto GPS por carreta/dia,2569,NaN,NaN,NaN,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,2025-09-30 21:01:32,2025-12-31 23:58:35,0.0,NaT,NaT,NaN,NaT,NaT,NaN
3,fato_readings,290535,8,uma leitura de odometro,10412,ID_LEITURA,290535.0,0.0,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,2020-01-01 07:00:23,2025-12-31 21:23:54,0.0,NaT,NaT,NaN
4,fato_wo,238818,12,uma ordem de servico,9860,ID_OS,238818.0,0.0,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,2020-01-01 09:15:10,2026-01-01 02:44:00,0.0
5,fato_wo_labour,759813,12,uma linha de mao de obra,9857,ID_LINHA_MAO_OBRA,759813.0,0.0,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,2020-01-01 09:15:10,2026-01-01 02:44:00,0.0
6,fato_wo_parts,738754,13,uma linha de peca,9728,ID_LINHA_PECA,738754.0,0.0,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,NaT,NaT,NaN,2020-01-01 12:31:00,2026-01-01 00:23:00,0.0


## Results

### 3. Confirmar relacionamento estrela

In [24]:
dim_ids = set(pd.read_csv(FILES["dim_carretas"], usecols=["ID_CARRETA"])["ID_CARRETA"])

integrity_rows = []
for name in [n for n in FILES if n != "dim_carretas"]:
    df = pd.read_csv(FILES[name], usecols=["ID_CARRETA"])
    ids = set(df["ID_CARRETA"].dropna())
    integrity_rows.append({
        "base": name,
        "carretas_distintas": len(ids),
        "ids_fora_dim_carretas": len(ids - dim_ids),
        "cobertura_sobre_dim": len(ids) / len(dim_ids),
    })

star_check = pd.DataFrame(integrity_rows)
star_check.to_csv(TABLES / "00_validacao_modelo_estrela.csv", index=False)
star_check

,base,carretas_distintas,ids_fora_dim_carretas,cobertura_sobre_dim
0,fato_contratos,10349,0,0.993949
1,fato_gps,2569,0,0.246735
2,fato_readings,10412,0,1.000000
3,fato_wo,9860,0,0.946984
4,fato_wo_labour,9857,0,0.946696
5,fato_wo_parts,9728,0,0.934307


### 4. Registrar riscos iniciais dos dados

In [25]:
wo = read_csv("fato_wo", usecols=["ID_OS", "DATA_OS", "TOTAL_INTERNO_MAO_OBRA", "TOTAL_INTERNO_PECAS"])
labour = read_csv("fato_wo_labour", usecols=["ID_OS", "COD_SISTEMA_VMRS", "SISTEMA_VMRS"])
gps = read_csv("fato_gps", usecols=["ID_CARRETA", "DATA_HORA_GPS"])
readings = read_csv("fato_readings", usecols=["ID_CARRETA", "KM_ACUMULADO", "KM_RESET_EM", "KM_RESET_PARA"])

wo["data_os"] = pd.to_datetime(wo["data_os"], errors="coerce")
for col in ["total_interno_mao_obra", "total_interno_pecas"]:
    wo[col] = as_number(wo[col])

gps["data_hora_gps"] = pd.to_datetime(gps["data_hora_gps"], errors="coerce")
labour["flag_linha_preventiva"] = (
    labour["cod_sistema_vmrs"].astype(str).str.upper().isin(PREVENTIVE_VMRS_CODES)
    | labour["sistema_vmrs"].astype(str).str.upper().str.contains("PREVENTIVE", na=False)
)

risks = pd.DataFrame([
    {
        "risco": "GPS com cobertura parcial",
        "evidencia": f"{gps['data_hora_gps'].min()} a {gps['data_hora_gps'].max()}, {gps['id_carreta'].nunique()} carretas",
        "tratamento_planejado": "usar como analise complementar, nao como feature central para todo o historico",
    },
    {
        "risco": "OS fora da janela declarada",
        "evidencia": f"{(wo['data_os'] >= pd.Timestamp('2026-01-01')).sum()} registros com DATA_OS >= 2026-01-01",
        "tratamento_planejado": "filtrar na base analitica mensal",
    },
    {
        "risco": "Custos negativos e zerados",
        "evidencia": f"{((wo['total_interno_mao_obra'] < 0) | (wo['total_interno_pecas'] < 0)).sum()} OS com algum custo negativo",
        "tratamento_planejado": "analisar como ajustes/estornos e filtrar apenas na modelagem se necessario",
    },
    {
        "risco": "Resets de odometro",
        "evidencia": f"{readings[['km_reset_em', 'km_reset_para']].notna().any(axis=1).sum()} leituras com campos de reset",
        "tratamento_planejado": "tratar deltas negativos ou anormais antes de calcular km do periodo",
    },
    {
        "risco": "Variavel-alvo ausente",
        "evidencia": "nao ha coluna pronta de custo por km nos CSVs brutos",
        "tratamento_planejado": "derivar custo por km no grao carreta x mes",
    },
    {
        "risco": "Alvo preventivo diferente de custo interno total",
        "evidencia": f"{labour['flag_linha_preventiva'].sum()} linhas de mao de obra classificadas como preventivas por VMRS PM/PREVENTIVE",
        "tratamento_planejado": "criar alvo primario custo_manutencao_preventiva_por_km e manter custo interno total como sensibilidade",
    },
    {
        "risco": "Tipo de manutencao contratual como confundidor",
        "evidencia": "MAINT/NET/MIX alteram a responsabilidade economica sobre manutencao",
        "tratamento_planejado": "usar MAINT como populacao principal de modelagem e reportar NET/MIX como segmentos/caveats",
    },
    {
        "risco": "Meses com baixa quilometragem distorcem razoes custo/km",
        "evidencia": f"piso metodologico definido: {KM_MIN_MES_ALVO:.0f} km/mes",
        "tratamento_planejado": "calcular alvos por km apenas para meses com km_rodado_mes >= piso",
    },
])

risks.to_csv(TABLES / "00_riscos_iniciais.csv", index=False)
risks

,risco,evidencia,tratamento_planejado
0,GPS com cobertura parcial,"2025-09-30 21:01:32 a 2025-12-31 23:58:35, 256...","usar como analise complementar, nao como featu..."
1,OS fora da janela declarada,3 registros com DATA_OS >= 2026-01-01,filtrar na base analitica mensal
2,Custos negativos e zerados,185 OS com algum custo negativo,analisar como ajustes/estornos e filtrar apena...
3,Resets de odometro,2505 leituras com campos de reset,tratar deltas negativos ou anormais antes de c...
4,Variavel-alvo ausente,nao ha coluna pronta de custo por km nos CSVs ...,derivar custo por km no grao carreta x mes
5,Alvo preventivo diferente de custo interno total,127684 linhas de mao de obra classificadas com...,criar alvo primario custo_manutencao_preventiv...
6,Tipo de manutencao contratual como confundidor,MAINT/NET/MIX alteram a responsabilidade econo...,usar MAINT como populacao principal de modelag...
7,Meses com baixa quilometragem distorcem razoes...,piso metodologico definido: 500 km/mes,calcular alvos por km apenas para meses com km...


## Takeaways

- As bases permitem montar uma visao mensal por carreta, integrando manutencao, odometro, contratos e atributos do ativo.
- `ID_CARRETA` e a chave estrutural do projeto; `ID_OS` detalha os custos de mao de obra e pecas.
- A preparacao precisa tratar qualidade de datas, resets de odometro, custos negativos/zerados, baixa quilometragem mensal e a cobertura parcial de GPS.